# So sanh tong hop cac phuong phap chon dac trung (v2)

Notebook nay tong hop ket qua tu baseline, kbest, rfe, model-based de dua ra bo dac trung cuoi cung cho buoc tuning.

## 1. Import thu vien

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Xac dinh duong dan va doc ket qua

In [ ]:
duong_dan_metrics_ung_vien = [
    Path("../../outputs/metrics"),
    Path("../outputs/metrics"),
    Path("outputs/metrics"),
    Path("AbaloneAge/outputs/metrics"),
]

duong_dan_metrics = None
for p in duong_dan_metrics_ung_vien:
    p_day_du = p.resolve()
    if p_day_du.exists():
        duong_dan_metrics = p_day_du
        break

if duong_dan_metrics is None:
    raise FileNotFoundError(
        "Khong tim thay thu muc outputs/metrics. Hay chay baseline, kbest, rfe, model-based truoc."
    )

tap_tin_can = {
    "baseline": duong_dan_metrics / "03_baseline_feature_scores.csv",
    "kbest": duong_dan_metrics / "03_kbest_feature_scores.csv",
    "rfe": duong_dan_metrics / "03_rfe_feature_ranking.csv",
    "model_based": duong_dan_metrics / "03_model_based_feature_ranking.csv",
}

for ten, p in tap_tin_can.items():
    if not p.exists():
        raise FileNotFoundError(f"Thieu file {ten}: {p}")

df_baseline = pd.read_csv(tap_tin_can["baseline"])
df_kbest = pd.read_csv(tap_tin_can["kbest"])
df_rfe = pd.read_csv(tap_tin_can["rfe"])
df_model = pd.read_csv(tap_tin_can["model_based"])

print("Thu muc metrics:", duong_dan_metrics)
print("Baseline  :", df_baseline.shape)
print("KBest     :", df_kbest.shape)
print("RFE       :", df_rfe.shape)
print("ModelBased:", df_model.shape)

## 3. Ham chuan hoa ten dac trung

In [ ]:
def chuan_hoa_ten_dac_trung(ten):
    t = str(ten)

    # Bo prefix do ColumnTransformer tao ra
    for prefix in ["so__", "hang_muc__one_hot__", "hang_muc__"]:
        if t.startswith(prefix):
            t = t[len(prefix):]

    # Gop one-hot gioi tinh ve mot bien tong quat 'sex'
    if t.startswith("sex_"):
        return "sex"

    return t

## 4. Tao diem cho tung phuong phap

In [ ]:
# Baseline
cot_ten_baseline = "đặc_trưng" if "đặc_trưng" in df_baseline.columns else "dac_trung"
cot_diem_baseline = "tương_quan" if "tương_quan" in df_baseline.columns else "tuong_quan"

bang_baseline = df_baseline[[cot_ten_baseline, cot_diem_baseline]].copy()
bang_baseline.columns = ["dac_trung_goc", "diem_goc"]
bang_baseline["dac_trung"] = bang_baseline["dac_trung_goc"].apply(chuan_hoa_ten_dac_trung)
bang_baseline = bang_baseline.groupby("dac_trung", as_index=False)["diem_goc"].max()
bang_baseline["baseline_score"] = bang_baseline["diem_goc"] / (bang_baseline["diem_goc"].max() + 1e-12)
bang_baseline = bang_baseline[["dac_trung", "baseline_score"]]

# KBest
bang_kbest = df_kbest[["dac_trung", "diem_f"]].copy()
bang_kbest["dac_trung"] = bang_kbest["dac_trung"].apply(chuan_hoa_ten_dac_trung)
bang_kbest = bang_kbest.groupby("dac_trung", as_index=False)["diem_f"].max()
bang_kbest["kbest_score"] = bang_kbest["diem_f"] / (bang_kbest["diem_f"].max() + 1e-12)
bang_kbest = bang_kbest[["dac_trung", "kbest_score"]]

# RFE
bang_rfe = df_rfe[["dac_trung", "duoc_chon", "xep_hang_rfe"]].copy()
bang_rfe["dac_trung"] = bang_rfe["dac_trung"].apply(chuan_hoa_ten_dac_trung)
max_rank = bang_rfe["xep_hang_rfe"].max()
bang_rfe["rfe_score_raw"] = np.where(
    bang_rfe["duoc_chon"] == True,
    (max_rank - bang_rfe["xep_hang_rfe"] + 1) / (max_rank + 1e-12),
    0.0,
)
bang_rfe = bang_rfe.groupby("dac_trung", as_index=False)["rfe_score_raw"].max()
bang_rfe["rfe_score"] = bang_rfe["rfe_score_raw"] / (bang_rfe["rfe_score_raw"].max() + 1e-12)
bang_rfe = bang_rfe[["dac_trung", "rfe_score"]]

# Model-based
bang_model = df_model[["dac_trung", "diem_trung_binh"]].copy()
bang_model["dac_trung"] = bang_model["dac_trung"].apply(chuan_hoa_ten_dac_trung)
bang_model = bang_model.groupby("dac_trung", as_index=False)["diem_trung_binh"].max()
bang_model["model_score"] = bang_model["diem_trung_binh"] / (bang_model["diem_trung_binh"].max() + 1e-12)
bang_model = bang_model[["dac_trung", "model_score"]]

## 5. Gop diem va xep hang tong

In [ ]:
bang_tong = bang_baseline.merge(bang_kbest, on="dac_trung", how="outer")
bang_tong = bang_tong.merge(bang_rfe, on="dac_trung", how="outer")
bang_tong = bang_tong.merge(bang_model, on="dac_trung", how="outer")

for c in ["baseline_score", "kbest_score", "rfe_score", "model_score"]:
    if c not in bang_tong.columns:
        bang_tong[c] = 0.0

bang_tong = bang_tong.fillna(0.0)
bang_tong["tong_diem"] = bang_tong[["baseline_score", "kbest_score", "rfe_score", "model_score"]].mean(axis=1)
bang_tong = bang_tong.sort_values(by="tong_diem", ascending=False).reset_index(drop=True)
bang_tong["xep_hang"] = np.arange(1, len(bang_tong) + 1)

bang_tong.head(15)

## 6. Truc quan hoa so sanh

In [ ]:
top_n = min(12, len(bang_tong))
bang_top = bang_tong.head(top_n).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bieu do cot tong diem
bang_bar = bang_top.sort_values(by="tong_diem")
axes[0].barh(bang_bar["dac_trung"], bang_bar["tong_diem"], color="slateblue")
axes[0].set_title("Top dac trung theo tong diem")
axes[0].set_xlabel("Tong diem")

# Heatmap diem theo phuong phap
du_lieu_heatmap = bang_top.set_index("dac_trung")[["baseline_score", "kbest_score", "rfe_score", "model_score"]]
sns.heatmap(du_lieu_heatmap, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1])
axes[1].set_title("Diem tung phuong phap")

plt.tight_layout()
duong_dan_hinh = (duong_dan_metrics.parent / "figures").resolve()
duong_dan_hinh.mkdir(parents=True, exist_ok=True)
plt.savefig(duong_dan_hinh / "03_feature_selection_compare_v2.png", dpi=300, bbox_inches="tight")
plt.show()

print("Da luu hinh: 03_feature_selection_compare_v2.png")

## 7. Chon bo dac trung de xuat

In [ ]:
so_dac_trung_de_xuat = min(6, len(bang_tong))
bo_de_xuat = bang_tong.head(so_dac_trung_de_xuat).copy()

print("Bo dac trung de xuat cuoi cung:")
for i, row in bo_de_xuat.iterrows():
    print(f"{int(row['xep_hang']):>2}. {row['dac_trung']:<20} | tong_diem={row['tong_diem']:.3f}")

bo_de_xuat

## 8. Doc metric tong ket de doi chieu (neu co)

In [ ]:
duong_dan_json = {
    "baseline": duong_dan_metrics / "03_baseline_summary.json",
    "kbest": duong_dan_metrics / "03_kbest_summary.json",
    "rfe": duong_dan_metrics / "03_rfe_summary.json",
    "model_based": duong_dan_metrics / "03_model_based_summary.json",
}

dong = []
for ten, p in duong_dan_json.items():
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)

        test = data.get("test", {})
        dong.append({
            "phuong_phap": ten,
            "MAE": test.get("MAE", np.nan),
            "RMSE": test.get("RMSE", np.nan),
            "R2": test.get("R2", np.nan),
        })

if len(dong) > 0:
    bang_metric = pd.DataFrame(dong).sort_values(by="RMSE")
    display(bang_metric)
else:
    bang_metric = pd.DataFrame(columns=["phuong_phap", "MAE", "RMSE", "R2"])
    print("Khong tim thay file summary json de doi chieu metric.")

## 9. Luu ket qua tong hop

In [ ]:
bang_tong.to_csv(duong_dan_metrics / "03_compare_v2_feature_scores.csv", index=False)
bo_de_xuat.to_csv(duong_dan_metrics / "03_compare_v2_selected_features.csv", index=False)

if len(bang_metric) > 0:
    bang_metric.to_csv(duong_dan_metrics / "03_compare_v2_method_metrics.csv", index=False)

tom_tat = {
    "phuong_phap": "compare_v2",
    "so_dac_trung_de_xuat": int(len(bo_de_xuat)),
    "dac_trung_de_xuat": bo_de_xuat["dac_trung"].tolist(),
    "top_10": bang_tong.head(10).to_dict(orient="records"),
}

with open(duong_dan_metrics / "03_compare_v2_summary.json", "w", encoding="utf-8") as f:
    json.dump(tom_tat, f, ensure_ascii=False, indent=2)

print("Da luu: 03_compare_v2_feature_scores.csv")
print("Da luu: 03_compare_v2_selected_features.csv")
if len(bang_metric) > 0:
    print("Da luu: 03_compare_v2_method_metrics.csv")
print("Da luu: 03_compare_v2_summary.json")